In [1]:
import pandas as pd

file_id = "1n2IJtB3CaaRTvR3Rs-r3Ipns0PI5uDsc"   # la partie entre /d/ et /view dans ton lien
url = f"https://drive.google.com/uc?id={file_id}"

df = pd.read_csv(url, sep=';')
display(df.head())

,LocalId,GlobalId,Age,Gender,Region,CityCode,TotalSpent,TotalPur,FirstPur,LastPur,...,CookPur,DiyPur,RefPur,ArtPur,GeogPur,ProvCook,ProvAtlas,ProvArt,RelPur,ProvPur
0,1,25,56.0,0,SW,M,297,5,16/07/2012,14,...,1,0,0,0,0,0,0,0,0,No
1,2,29,23.0,1,NE,S,128,4,13/07/2013,8,...,0,0,0,0,0,0,0,0,0,No
2,3,46,61.0,0,NE,D,138,10,04/09/2009,22,...,2,0,1,0,1,1,0,0,2,No
3,4,47,30.0,0,SW,G,228,2,09/03/2014,2,...,0,0,0,0,0,0,0,0,0,No
4,5,51,NaN,0,SE,C,257,3,09/07/2013,10,...,0,0,0,0,0,0,0,0,0,No


### Checking the Full Dataset Size and Displaying More Rows

By default, `df.head()` only shows the first 5 rows. Let's print the actual shape of your dataset and display the first 20 rows instead.

In [2]:
# Print the full dimensions of the DataFrame (rows, columns)
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}\n")

# Display the first 20 rows of your data
display(df.head(20))

Total rows: 4000
Total columns: 23



,LocalId,GlobalId,Age,Gender,Region,CityCode,TotalSpent,TotalPur,FirstPur,LastPur,...,CookPur,DiyPur,RefPur,ArtPur,GeogPur,ProvCook,ProvAtlas,ProvArt,RelPur,ProvPur
0,1,25,56.0,0,SW,M,297,5,16/07/2012,14,...,1,0,0,0,0,0,0,0,0,No
1,2,29,23.0,1,NE,S,128,4,13/07/2013,8,...,0,0,0,0,0,0,0,0,0,No
2,3,46,61.0,0,NE,D,138,10,04/09/2009,22,...,2,0,1,0,1,1,0,0,2,No
3,4,47,30.0,0,SW,G,228,2,09/03/2014,2,...,0,0,0,0,0,0,0,0,0,No
4,5,51,NaN,0,SE,C,257,3,09/07/2013,10,...,0,0,0,0,0,0,0,0,0,No
5,6,60,29.0,0,SE,B,145,4,10/05/2013,6,...,0,0,0,0,0,0,0,0,0,No
6,7,61,61.0,0,NE,K,190,3,13/01/2013,16,...,0,0,0,0,1,0,0,0,1,No
7,8,79,58.0,0,SW,P,187,4,26/03/2013,14,...,0,0,0,0,0,0,0,0,0,No
8,9,81,41.0,0,SE,R,252,4,16/07/2013,10,...,0,0,0,0,0,0,0,0,0,No
9,10,90,71.0,0,NE,F,240,5,20/09/2012,6,...,1,0,0,0,0,0,0,0,0,No


preparation des donnees

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

CIBLE = "ProvPur"   # variable de classe

# Date de premier achat -> date, puis ancienneté en mois
df["FirstPur"] = pd.to_datetime(df["FirstPur"], format="%d/%m/%Y", errors="coerce")
df["AncienneteMois"] = ((df["FirstPur"].max() - df["FirstPur"]).dt.days / 30.44).round(1)

IDENTIFIANTS = ["LocalId", "GlobalId"]
QUALITATIVES = ["Gender", "Region", "CityCode", CIBLE]
for c in QUALITATIVES:
    df[c] = df[c].astype("category")
QUANTITATIVES = [c for c in df.select_dtypes(include="number").columns if c not in IDENTIFIANTS]

print("Dimensions :", df.shape)
print("Quantitatives :", QUANTITATIVES)
print("Qualitatives :", QUALITATIVES)
df.info()

Dimensions : (4000, 24)
Quantitatives : ['Age', 'TotalSpent', 'TotalPur', 'LastPur', 'NovelPur', 'ChildPur', 'YouthPur', 'CookPur', 'DiyPur', 'RefPur', 'ArtPur', 'GeogPur', 'ProvCook', 'ProvAtlas', 'ProvArt', 'RelPur', 'AncienneteMois']
Qualitatives : ['Gender', 'Region', 'CityCode', 'ProvPur']
<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   LocalId         4000 non-null   int64         
 1   GlobalId        4000 non-null   int64         
 2   Age             3827 non-null   float64       
 3   Gender          4000 non-null   category      
 4   Region          4000 non-null   category      
 5   CityCode        3903 non-null   category      
 6   TotalSpent      4000 non-null   int64         
 7   TotalPur        4000 non-null   int64         
 8   FirstPur        4000 non-null   datetime64[us]
 9   LastPur         4000 non-null  

valeur manquant et doublons


In [4]:
manq = pd.DataFrame({"n": df.isna().sum(), "%": (df.isna().mean()*100).round(2)})
display(manq[manq["n"] > 0])
print("Doublons (hors identifiants) :", df.drop(columns=IDENTIFIANTS).duplicated().sum())

,n,%
Age,173,4.32
CityCode,97,2.43


Doublons (hors identifiants) : 0


Vérification et analyse de la colonne 'Age'

Regardons de plus près la distribution de la colonne `Age` pour vérifier s'il y a des valeurs suspectes (comme un âge de 173 ans) et observons les premières lignes contenant des valeurs manquantes (`NaN`).

In [5]:
# 1. Statistiques descriptives de l'âge
print("--- Statistiques de la colonne Age ---")
print(df['Age'].describe())

# 2. Aperçu des premières lignes avec un Age manquant (NaN)
print("\n--- Aperçu des lignes avec Age manquant (NaN) ---")
display(df[df['Age'].isna()].head(10))

--- Statistiques de la colonne Age ---
count    3827.000000
mean       44.645937
std        16.003261
min        12.000000
25%        31.000000
50%        48.000000
75%        58.000000
max        87.000000
Name: Age, dtype: float64

--- Aperçu des lignes avec Age manquant (NaN) ---


,LocalId,GlobalId,Age,Gender,Region,CityCode,TotalSpent,TotalPur,FirstPur,LastPur,NovelPur,ChildPur,YouthPur,CookPur,DiyPur,RefPur,ArtPur,GeogPur,ProvCook,ProvAtlas,ProvArt,RelPur,ProvPur,AncienneteMois
4,5,51,NaN,0,SE,C,257,3,2013-07-09,10,2,0,0,0,0,0,0,0,0,0,0,0,No,8.6
10,11,95,NaN,0,NE,Q,275,5,2012-09-05,2,1,0,0,1,0,0,0,0,0,0,0,0,No,18.7
15,16,121,NaN,0,SW,M,256,12,2009-07-09,14,3,2,0,3,0,1,1,2,1,1,1,6,No,56.6
42,43,345,NaN,0,NE,Q,234,5,2013-07-08,4,3,0,0,0,0,0,0,0,0,0,0,0,No,8.6
74,75,594,NaN,0,SW,M,193,3,2012-09-10,16,1,1,0,0,0,0,1,0,0,0,0,1,No,18.5
87,88,712,NaN,0,SE,A,306,4,2013-01-16,10,2,0,0,0,0,0,1,0,0,0,0,1,No,14.3
90,91,743,NaN,0,SW,M,240,4,2012-09-21,20,3,1,0,0,0,0,0,0,0,0,0,0,No,18.2
111,112,954,NaN,0,NE,E,137,2,2013-01-25,16,1,0,0,0,0,1,0,0,0,0,0,0,No,14.0
159,160,1329,NaN,0,SW,M,163,1,2014-03-05,2,0,0,0,0,0,0,0,0,0,0,0,0,No,0.8
166,167,1371,NaN,0,NE,J,295,4,2011-01-20,32,2,2,0,0,0,0,0,0,0,0,0,0,No,38.2


statistique descriptive

In [6]:
def resume(s):
    s = s.dropna()
    return pd.Series({
        "moyenne": s.mean(), "écart-type": s.std(), "min": s.min(),
        "Q1": s.quantile(.25), "médiane": s.median(), "Q3": s.quantile(.75),
        "max": s.max(), "IQR": s.quantile(.75) - s.quantile(.25),
        "nb valeurs": s.nunique(),
    })

desc = df[QUANTITATIVES].apply(resume).T.round(2)
display(desc)



# Valeurs aberrantes (règle 1,5 x IQR) + test de normalité
lignes = []
for c in QUANTITATIVES:
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    n_out = ((df[c] < q1 - 1.5*iqr) | (df[c] > q3 + 1.5*iqr)).sum()
    p = stats.normaltest(df[c].dropna()).pvalue
    lignes.append({"variable": c, "nb aberrantes": n_out,
               3    "% aberrantes": round(n_out/df[c].count()*100, 1),
                   "p normalité": p, "normale ?": p > 0.05})
display(pd.DataFrame(lignes))

SyntaxError: ':' expected after dictionary key (2050747631.py, line 22)

In [ ]:
n = len(QUANTITATIVES); ncol = 4; nrow = int(np.ceil(n / ncol))

# 1. Distribution des variables quantitatives (Histogrammes)
fig, axes = plt.subplots(nrow, ncol, figsize=(4.5*ncol, 3.3*nrow))
for ax, c in zip(axes.flat, QUANTITATIVES):
    discret = df[c].nunique() <= 25
    sns.histplot(df[c].dropna(), ax=ax, kde=not discret, discrete=discret, color="steelblue")
    ax.axvline(df[c].mean(), color="red", ls="--", label="moyenne")
    ax.axvline(df[c].median(), color="green", ls=":", label="médiane")
    ax.set_title(c); ax.set_xlabel("")

for ax in axes.flat[n:]: ax.set_visible(False)
axes.flat[0].legend()
fig.suptitle("Distribution des variables quantitatives", fontsize=14)
plt.tight_layout(); plt.show()

# 2. Premiers achats dans le temps
plt.figure(figsize=(10, 4))
df["FirstPur"].dt.to_period("M").value_counts().sort_index().plot()
plt.title("Nombre de clients par mois de premier achat")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## Analyses Bivariées : Qui sont les acheteurs (`ProvPur` = Yes vs No) ?

Pour répondre à la question du professeur, analysons comment les comportements et profils diffèrent entre ceux qui achètent localement (`ProvPur == 'Yes'`) et ceux qui n'achètent pas (`ProvPur == 'No'`).

In [ ]:
# 1. Taux de conversion global
taux_conversion = df['ProvPur'].value_counts(normalize=True) * 100
print("--- Répartition de la cible ProvPur --- ")
print(taux_conversion.round(2))

# 2. Comparaison des profils quantitatifs (Moyennes)
print("\n--- Profil moyen des acheteurs vs non-acheteurs ---")
profil_moyen = df.groupby('ProvPur', observed=False)[['Age', 'TotalSpent', 'TotalPur', 'AncienneteMois']].mean().round(2)
display(profil_moyen)

In [ ]:
# 3. Visualisation de la différence d'âge et de dépenses
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='ProvPur', y='Age', ax=axes[0], palette='Pastel1', hue='ProvPur', legend=False)
axes[0].set_title('Distribution de l\'Âge selon l\'Achat (ProvPur)')

sns.boxplot(data=df, x='ProvPur', y='TotalSpent', ax=axes[1], palette='Pastel2', hue='ProvPur', legend=False)
axes[1].set_title('Distribution des Dépenses selon l\'Achat (ProvPur)')

plt.tight_layout()
plt.show()

In [ ]:
# 4. Analyse des variables qualitatives (Sexe et Région)
print("--- Taux d'achat (ProvPur = Yes) par Genre ---")
gender_cross = pd.crosstab(df['Gender'], df['ProvPur'], normalize='index') * 100
display(gender_cross.round(2))

print("\n--- Taux d'achat (ProvPur = Yes) par Région ---")
region_cross = pd.crosstab(df['Region'], df['ProvPur'], normalize='index') * 100
display(region_cross.round(2))

### Analyse statistique des variables qualitatives (Test du Chi-deux et Visualisations)

Pour déterminer si le genre ou la région influencent de manière significative l'achat provincial (`ProvPur`), nous réalisons des tests d'indépendance du $\chi^2$ (Chi-deux) et visualisons les taux de conversion.

In [ ]:
# Tests de Chi-deux pour mesurer l'association statistique
for col in ['Gender', 'Region']:
    contingency_table = pd.crosstab(df[col], df['ProvPur'])
    chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
    print(f"--- Test du Chi-deux pour {col} vs ProvPur ---")
    print(f"Statistique Chi2 : {chi2:.4f}")
    print(f"p-value : {p:.4e}")
    if p < 0.05:
        print(f"=> Relation STATISTIQUEMENT SIGNIFICATIVE au seuil de 5% (p < 0.05)\n")
    else:
        print(f"=> Pas de relation statistiquement significative (p >= 0.05)\n")

In [ ]:
# Visualisation des profils d'achat par Genre et par Région
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Graphique par Genre
gender_prop = pd.crosstab(df['Gender'], df['ProvPur'], normalize='index')
gender_prop.plot(kind='bar', stacked=True, ax=axes[0], color=['#ff9999','#66b3ff'])
axes[0].set_title("Proportion d'achat selon le Genre")
axes[0].set_ylabel("Proportion")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# 2. Graphique par Région
region_prop = pd.crosstab(df['Region'], df['ProvPur'], normalize='index')
region_prop.plot(kind='bar', stacked=True, ax=axes[1], color=['#ff9999','#66b3ff'])
axes[1].set_title("Proportion d'achat selon la Région")
axes[1].set_ylabel("Proportion")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

## Analyse de l'Achat Provincial (`ProvPur` et Volume) par Tranche d'Âge

Créons des tranches d'âge pour observer si certaines générations sont plus enclines à réaliser des achats provinciaux.

In [ ]:
# 1. Création de tranches d'âge de 10 ans
df['Tranche_Age'] = pd.cut(df['Age'], bins=[10, 20, 30, 40, 50, 60, 70, 80, 90],
                             labels=['10-20', '20-30', '30-40', '40-50', '50-60', '60-70', '70-80', '80-90'])

# 2. Agrégation : Garder uniquement le nombre de clients par tranche d'âge
analyse_age_clients = df.groupby('Tranche_Age', observed=False).agg(
    Nombre_Clients=('ProvPur', 'count')
)

display(analyse_age_clients)

# 3. Histogramme simple montrant uniquement le nombre de clients par tranche d'âge (sans le taux)
plt.figure(figsize=(10, 5))
sns.barplot(data=analyse_age_clients.reset_index(), x='Tranche_Age', y='Nombre_Clients', color='skyblue', edgecolor='black')
plt.title('Répartition du nombre de clients par tranche d\'âge')
plt.xlabel('Tranche d\'âge')
plt.ylabel('Nombre de clients')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Analyse de la dépendance des variables qualitatives (Test d'association statistique et Visualisations)

Pour déterminer si le genre ou la région influencent de manière significative l'achat provincial (`ProvPur`), nous réalisons des tests d'association statistique (tests d'indépendance) et visualisons les taux de conversion.

In [ ]:
import pandas as pd
from scipy import stats

# Tests d'association statistique pour mesurer la dépendance entre les variables
for col in ['Gender', 'Region']:
    contingency_table = pd.crosstab(df[col], df['ProvPur'])
    chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
    print(f"--- Test d'association pour {col} vs ProvPur ---")
    print(f"Statistique du test : {chi2:.4f}")
    print(f"p-value : {p:.4e}")
    if p < 0.05:
        print(f"=> Relation STATISTIQUEMENT SIGNIFICATIVE au seuil de 5% (p < 0.05)\n")
    else:
        print(f"=> Pas de relation statistiquement significative (p >= 0.05)\n")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Visualisation des profils d'achat par Genre et par Région
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Graphique par Genre
gender_prop = pd.crosstab(df['Gender'], df['ProvPur'], normalize='index')
gender_prop.plot(kind='bar', stacked=True, ax=axes[0], color=['#ff9999','#66b3ff'])
axes[0].set_title("Proportion d'achat selon le Genre")
axes[0].set_ylabel("Proportion")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# 2. Graphique par Région
region_prop = pd.crosstab(df['Region'], df['ProvPur'], normalize='index')
region_prop.plot(kind='bar', stacked=True, ax=axes[1], color=['#ff9999','#66b3ff'])
axes[1].set_title("Proportion d'achat selon la Région")
axes[1].set_ylabel("Proportion")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()